# RUTINA DESPIECES

In [1]:
import ezdxf
import json, math, os
import ipywidgets as widgets
from IPython.display import display

## 2. Cargar y leer el JSON

In [2]:
# Cargar el JSON — misma carpeta que el notebook
with open("vigas.json", encoding="utf-8") as f:
    datos = json.load(f)

elementos = datos["elementos"]

print(f"Proyecto : {datos['proyecto']}")
print(f"Fecha    : {datos['fecha']}")
print(f"Elementos: {len(elementos)}")
print()
print("Primeros 3 elementos:")
for e in elementos[:3]:
    print(f"  {e['marca']} | {e['nivel']} | {e['seccion']} | {e['longitud']}m")

Proyecto : PROYECTO_INICIAL
Fecha    : 2026-03-18
Elementos: 37

Primeros 3 elementos:
  VG-380040 | 00_Base | V40X50 | 5.0m
  VG-380147 | 00_Base | V40X50 | 5.0m
  VG-380183 | 00_Base | V40X50 | 5.0m


## 3. Funciones de encadenamiento

In [3]:
TOL = 0.15
TOL_ENCADENAMIENTO = 0.25  # tolerancia mayor solo para encadenar extremos

def dist2d(p1, p2):
    return math.sqrt((p1["x"]-p2["x"])**2 + (p1["y"]-p2["y"])**2)

def coinciden(p1, p2):
    return dist2d(p1, p2) < TOL_ENCADENAMIENTO  # usar tolerancia mayor

def colineales(e1, e2):
    dx1 = e1["fin"]["x"] - e1["inicio"]["x"]
    dy1 = e1["fin"]["y"] - e1["inicio"]["y"]
    l1  = math.sqrt(dx1**2 + dy1**2)
    if l1 < 1e-6: return False
    dx2 = e2["fin"]["x"] - e2["inicio"]["x"]
    dy2 = e2["fin"]["y"] - e2["inicio"]["y"]
    l2  = math.sqrt(dx2**2 + dy2**2)
    if l2 < 1e-6: return False
    return abs(dx1/l1 * dy2/l2 - dy1/l1 * dx2/l2) < 0.05

def encadenar(elems):
    usados = set()
    cadenas = []
    for i, e in enumerate(elems):
        if i in usados:
            continue
        cadena = [e]
        usados.add(i)
        cambio = True
        while cambio:
            cambio = False
            for j, o in enumerate(elems):
                if j in usados: continue
                if o["nivel"]   != cadena[0]["nivel"]:   continue
                if o["seccion"] != cadena[0]["seccion"]: continue
                if not colineales(cadena[0], o):         continue
                ef = cadena[-1]["fin"]
                ei = cadena[0]["inicio"]
                if coinciden(o["inicio"], ef):
                    cadena.append(o); usados.add(j); cambio = True
                elif coinciden(o["fin"], ei):
                    cadena.insert(0, o); usados.add(j); cambio = True
                elif coinciden(o["fin"], ef):
                    cadena.append({**o, "inicio": o["fin"], "fin": o["inicio"]})
                    usados.add(j); cambio = True
                elif coinciden(o["inicio"], ei):
                    cadena.insert(0, {**o, "inicio": o["fin"], "fin": o["inicio"]})
                    usados.add(j); cambio = True
        cadenas.append(cadena)
    return cadenas

print(f"✓ Funciones definidas — TOL encadenamiento={TOL_ENCADENAMIENTO}m")

✓ Funciones definidas — TOL encadenamiento=0.25m


## 4. Ejecutar encadenamiento y agrupar


In [4]:
def normalizar(vanos):
    return tuple(min(tuple(vanos), tuple(reversed(vanos))))

cadenas = encadenar(elementos)

grupos = {}
for cadena in cadenas:
    nivel   = cadena[0]["nivel"]
    seccion = cadena[0]["seccion"]
    vanos   = [round(e["longitud"], 3) for e in cadena]
    clave   = (nivel, seccion, normalizar(vanos))
    if clave not in grupos:
        grupos[clave] = {
            "nivel":    nivel,
            "seccion":  seccion,
            "vanos":    vanos,
            "cantidad": 0,
            "total":    round(sum(vanos), 3)
        }
    grupos[clave]["cantidad"] += 1

# Ordenar y asignar marca automática VG-01 reiniciando por nivel
vigas = sorted(grupos.values(), key=lambda v: (v["nivel"], v["seccion"], -v["cantidad"]))

nivel_actual = None
contador = 0
for v in vigas:
    if v["nivel"] != nivel_actual:
        nivel_actual = v["nivel"]
        contador = 1
    else:
        contador += 1
    v["marca"] = f"VG-{contador:02d}"

# Mostrar resumen
print(f"Elementos entrada : {len(elementos)}")
print(f"Cadenas detectadas: {len(cadenas)}")
print(f"Tipos únicos      : {len(vigas)}")
print()

nivel_actual = None
for v in vigas:
    if v["nivel"] != nivel_actual:
        nivel_actual = v["nivel"]
        print(f"NIVEL: {nivel_actual}")
    vanos_str = " + ".join([f"{x}m" for x in v["vanos"]])
    print(f"  {v['marca']}  {v['seccion']:10} | {vanos_str} | total: {v['total']}m | cant: {v['cantidad']}")

Elementos entrada : 37
Cadenas detectadas: 33
Tipos únicos      : 9

NIVEL: 00_Base
  VG-01  V40X50     | 5.0m | total: 5.0m | cant: 3
  VG-02  V40X50     | 5.0m + 3.65m | total: 8.65m | cant: 2
NIVEL: 01_Piso2
  VG-01  V15x50     | 5.45m | total: 5.45m | cant: 5
  VG-02  V40X50     | 5.0m | total: 5.0m | cant: 3
  VG-03  V40X50     | 5.0m + 3.65m | total: 8.65m | cant: 2
NIVEL: 02_Piso3
  VG-01  V15x50     | 5.5m | total: 5.5m | cant: 5
  VG-02  V40X50     | 5.0m | total: 5.0m | cant: 4
NIVEL: 03_Cub
  VG-01  V15x50     | 5.5m | total: 5.5m | cant: 5
  VG-02  V40X50     | 5.0m | total: 5.0m | cant: 4


## 5. Función que dibuja una viga en DXF

In [5]:
ALTURA_VIGA    = 2.15
ANCHO_NOMBRE   = 2.0
ALTO_NOMBRE    = 2.15
ALTURA_TEXTO   = 0.25
ALTURA_TEXTO_S = 0.15
ANCHO_COLUMNA  = 0.30

def dibujar_viga(msp, viga, x_inicio, y_inicio):
    x = x_inicio
    y = y_inicio
    medio_col = ANCHO_COLUMNA / 2

    # --- Bloque encabezado ---
    msp.add_lwpolyline(
        [(x, y), (x + ANCHO_NOMBRE, y),
         (x + ANCHO_NOMBRE, y + ALTO_NOMBRE), (x, y + ALTO_NOMBRE)],
        close=True
    )
    cx = x + ANCHO_NOMBRE / 2
    msp.add_text(viga["marca"],
        dxfattribs={"height": ALTURA_TEXTO, "insert": (cx, y + 1.65),
                    "halign": 4, "valign": 0, "align_point": (cx, y + 1.65)})
    msp.add_text(viga["nivel"],
        dxfattribs={"height": ALTURA_TEXTO_S, "insert": (cx, y + 0.80),
                    "halign": 4, "valign": 0, "align_point": (cx, y + 0.80)})
    msp.add_text(f"({viga['seccion']})",
        dxfattribs={"height": ALTURA_TEXTO_S, "insert": (cx, y + 0.60),
                    "halign": 4, "valign": 0, "align_point": (cx, y + 0.60)})
    msp.add_text(f"Es {viga['cantidad']}",
        dxfattribs={"height": ALTURA_TEXTO_S, "insert": (cx, y + 0.40),
                    "halign": 4, "valign": 0, "align_point": (cx, y + 0.40)})

    x += ANCHO_NOMBRE

    # --- Primera columna (apoyo inicio) ---
    msp.add_lwpolyline(
        [(x, y), (x + ANCHO_COLUMNA, y),
         (x + ANCHO_COLUMNA, y + ALTURA_VIGA), (x, y + ALTURA_VIGA)],
        close=True
    )
    # Línea central de la columna
    msp.add_line((x + medio_col, y), (x + medio_col, y + ALTURA_VIGA))
    x += ANCHO_COLUMNA

    # --- Vanos y columnas intermedias/final ---
    for idx, longitud in enumerate(viga["vanos"]):
        # Longitud neta del vano = centro a centro - ancho columna
        longitud_neta = longitud - ANCHO_COLUMNA

        # Rectángulo del vano neto
        msp.add_lwpolyline(
            [(x, y), (x + longitud_neta, y),
             (x + longitud_neta, y + ALTURA_VIGA), (x, y + ALTURA_VIGA)],
            close=True
        )
        x += longitud_neta

        # Columna al final de este vano
        msp.add_lwpolyline(
            [(x, y), (x + ANCHO_COLUMNA, y),
             (x + ANCHO_COLUMNA, y + ALTURA_VIGA), (x, y + ALTURA_VIGA)],
            close=True
        )
        # Línea central de la columna
        msp.add_line((x + medio_col, y), (x + medio_col, y + ALTURA_VIGA))
        x += ANCHO_COLUMNA

    return ALTO_NOMBRE + 0.5

print(f"✓ Función actualizada — columna: {ANCHO_COLUMNA}m")

✓ Función actualizada — columna: 0.3m


# 6. Previsualizar viga

In [6]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

def previsualizar_despiece(vigas_lista):
    medio_col = ANCHO_COLUMNA / 2
    fig, ax = plt.subplots(figsize=(18, len(vigas_lista) * 1.5 + 1))
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"Despiece de Vigas — {datos['proyecto']}", fontsize=12, fontweight="bold")

    y = 0
    for viga in vigas_lista:
        x = 0
        # --- Encabezado ---
        ax.add_patch(patches.Rectangle((x, y), ANCHO_NOMBRE, ALTO_NOMBRE,
                     linewidth=0.8, edgecolor="black", facecolor="#f0f0f0"))
        ax.text(x + ANCHO_NOMBRE/2, y + 1.65, viga["marca"],
                ha="center", va="center", fontsize=7, fontweight="bold")
        ax.text(x + ANCHO_NOMBRE/2, y + 0.80, viga["nivel"],
                ha="center", va="center", fontsize=5.5, color="#444")
        ax.text(x + ANCHO_NOMBRE/2, y + 0.60, f"({viga['seccion']})",
                ha="center", va="center", fontsize=5.5, color="#444")
        ax.text(x + ANCHO_NOMBRE/2, y + 0.40, f"Es {viga['cantidad']}",
                ha="center", va="center", fontsize=5.5, color="#444")
        x += ANCHO_NOMBRE

        # --- Primera columna ---
        ax.add_patch(patches.Rectangle((x, y), ANCHO_COLUMNA, ALTURA_VIGA,
                     linewidth=0.8, edgecolor="black", facecolor="#cccccc"))
        ax.plot([x + medio_col, x + medio_col], [y, y + ALTURA_VIGA], "k--", linewidth=0.6)
        x += ANCHO_COLUMNA

        # --- Vanos ---
        for longitud in viga["vanos"]:
            longitud_neta = longitud - ANCHO_COLUMNA
            ax.add_patch(patches.Rectangle((x, y), longitud_neta, ALTURA_VIGA,
                         linewidth=0.8, edgecolor="black", facecolor="white"))
            ax.annotate("", xy=(x + longitud_neta, y + ALTURA_VIGA + 0.3),
                        xytext=(x, y + ALTURA_VIGA + 0.3),
                        arrowprops=dict(arrowstyle="<->", color="black", lw=0.6))
            ax.text(x + longitud_neta/2, y + ALTURA_VIGA + 0.5,
                    f"{longitud}m", ha="center", va="center", fontsize=6)
            x += longitud_neta
            ax.add_patch(patches.Rectangle((x, y), ANCHO_COLUMNA, ALTURA_VIGA,
                         linewidth=0.8, edgecolor="black", facecolor="#cccccc"))
            ax.plot([x + medio_col, x + medio_col], [y, y + ALTURA_VIGA], "k--", linewidth=0.6)
            x += ANCHO_COLUMNA

        y -= (ALTO_NOMBRE + 3.0)

    ax.autoscale()
    plt.tight_layout()
    plt.show()

# --- Widget interactivo ---
vigas_filtradas = [v for v in vigas if "15x" not in v["seccion"].lower()]

opciones = [(f"{v['marca']} | {v['nivel']} | {v['seccion']} | {'+'.join([str(x)+'m' for x in v['vanos']])} | cant:{v['cantidad']}", i)
            for i, v in enumerate(vigas_filtradas)]

selector = widgets.Dropdown(
    options=opciones,
    description="Viga:",
    layout=widgets.Layout(width="600px")
)

output = widgets.Output()

def on_change(change):
    with output:
        output.clear_output(wait=True)
        previsualizar_despiece([vigas_filtradas[change["new"]]])

selector.observe(on_change, names="value")

with output:
    previsualizar_despiece([vigas_filtradas[0]])

display(selector, output)

Dropdown(description='Viga:', layout=Layout(width='600px'), options=(('VG-01 | 00_Base | V40X50 | 5.0m | cant:…

Output()

## 6. Generar DXF

In [7]:
# Crear el documento DXF
doc = ezdxf.new()
doc.header["$INSUNITS"] = 6  # 6 = metros
msp = doc.modelspace()

# Posición inicial
X_BASE = 0.0
Y_BASE = 0.0
SEPARACION = 2.5  # espacio vertical entre vigas

y_actual = Y_BASE
nivel_actual = None

for viga in vigas:
    # Saltar viguetas (sección contiene "15x")
    if "15x" in viga["seccion"].lower():
        continue
    
    # Separador extra entre niveles
    if viga["nivel"] != nivel_actual:
        if nivel_actual is not None:
            y_actual -= 2.0
        nivel_actual = viga["nivel"]

    alto_usado = dibujar_viga(msp, viga, X_BASE, y_actual)
    y_actual -= (alto_usado + SEPARACION)

# Guardar
ruta_dxf = "despiece_vigas.dxf"
doc.saveas(ruta_dxf)
print(f"✓ DXF generado: {ruta_dxf}")
print(f"  Vigas dibujadas: {len(vigas)}")

✓ DXF generado: despiece_vigas.dxf
  Vigas dibujadas: 9
